In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from pyspark.sql.types import *

In [0]:
df_raw = spark.read.table('ipl.s1_raw.ipl_data')
display(df_raw)

In [0]:
df_innings = df_raw.select(posexplode(df_raw.innings).alias("innings_num","innings"),"match_id","team1","team2")

df_innings = df_innings.select\
                (
                    col("match_id"),
                    concat(col("match_id"),lit("_"),col("innings_num")+1).alias("innings_id"),
                    (col("innings_num")+1).alias("innings_num"),
                    col("innings.team").alias("batting_team"),
                    when(col("innings.team")==col("team1"),col("team2")).otherwise(col("team1")).alias("fielding_team"),
                    size(df_innings.innings.overs).alias("overs_cnt"),
                    size(df_innings.innings.overs[size(df_innings.innings.overs)-1].deliveries).alias("balls"),
                    df_innings.innings.overs[size(df_innings.innings.overs)-1].deliveries.alias("deliveries")
                )

display(df_innings)

df_innings = df_innings.select\
                (
                    col("match_id"),
                    col("innings_id"),
                    col("innings_num"),
                    col("batting_team"),
                    col("fielding_team"),
                    col("overs_cnt").cast(DecimalType(10, 1))
                )




In [0]:
df_innings.write.mode("overwrite").saveAsTable("ipl.s2_siver.innings")